In [174]:
import tensorflow as tf
from tqdm import tqdm_notebook

print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [175]:
# ============================================================
# 0) INSTALL
# ============================================================
# pip install -U LibRecommender tensorflow pandas numpy clickhouse-connect scikit-learn

# If you are in notebook / interactive env and hit TF graph reuse issues:
# import tensorflow as tf
# tf.compat.v1.reset_default_graph()


# ============================================================
# 1) IMPORTS
# ============================================================
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import clickhouse_connect
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from recora.data import DatasetFeat

import pandas as pd
import tensorflow as tf

from recora.algorithms import (
	DIN,
	DeepFM,
	TwoTower,
	YouTubeRetrieval
)
from recora.data import DatasetFeat, split_by_ratio_chrono

TRAIN_DAYS = 90
RANDOM_SEED = 42
TOP_K = 20

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

warnings.filterwarnings("ignore")

In [176]:
# ============================================================
# 2) GLOBAL CONFIG
# ============================================================
CLICKHOUSE_CONFIG = {
	"host": os.getenv("CH_HOST", "172.30.36.13"),
	"port": int(os.getenv("CH_PORT", "8123")),
	"username": os.getenv("CH_USER", "default"),
	"password": os.getenv("CH_PASSWORD", "192837465AhurA!@#"),
	"database": os.getenv("CH_DATABASE", "KF"),
	"secure": os.getenv("CH_SECURE", "false").lower() == "true",
}

# ============================================================
# 3) CLICKHOUSE CLIENT
# ============================================================
client = clickhouse_connect.get_client(**CLICKHOUSE_CONFIG)

# # ============================================================
# # 4) SQL QUERIES
# #    KEEPING YOUR SQL UNCHANGED
# # ============================================================
# ITEM_SQL = f"""
# /*----- CTEs for item features -----*/
#
# WITH item_base AS (SELECT pp.brand_id           AS brand_id,
#                           pp.category_level2_id AS category_id,
#                           pp.category_level1_id AS root_category_id
#                    FROM KF.plane_products pp
#                    WHERE pp.category_level1_activity_type_id IN (1, 29)
#                      AND pp.category_level2_activity_type_id IN (1, 29)
#                      AND pp.product_id <> 0
#                    GROUP BY pp.brand_id, pp.category_level2_id, pp.category_level1_id),
#
#      availability_price AS (SELECT pp.brand_id                    AS brand_id,
#                                    pp.category_level2_id          AS category_id,
#                                    avg(isph.availability)         AS avg_availability,
#                                    stddevSamp(isph.availability)  AS std_availability,
#
#                                    avg(isph.avg_price)            AS avg_price,
#                                    stddevSamp(isph.avg_price)     AS std_price,
#                                    min(isph.avg_price)            AS min_price,
#                                    max(isph.avg_price)            AS max_price,
#                                    quantile(0.25)(isph.avg_price) AS p25_price,
#                                    quantile(0.5)(isph.avg_price)  AS p50_price, -- corrected: median
#                                    quantile(0.75)(isph.avg_price) AS p75_price
#                             FROM KF.in_stock_price_history isph
#                                      INNER JOIN KF.plane_products pp ON isph.shop_product_id = pp.shop_product_id
#                             WHERE isph.snapshot_date > today() - {TRAIN_DAYS}
#                               AND pp.category_level1_activity_type_id IN (1, 29)
#                               AND pp.category_level2_activity_type_id IN (1, 29)
#                               AND pp.product_id <> 0
#                             GROUP BY pp.brand_id, pp.category_level2_id),
#
#      popularity_discount AS (SELECT pp.brand_id                                                  AS brand_id,
#                                     pp.category_level2_id                                        AS category_id,
#
#                                     count(*)                                                     AS total_popularity,
#                                     countIf(oi.order_date > today() - 30)                        AS recent_popularity,
#                                     recent_popularity / total_popularity                         AS popularity_trend,
#
#                                     sum(oi.items_jet_discount + oi.brand_discount + oi.items_vendor_discount) /
#                                     nullIf(sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount +
#                                                oi.brand_discount + oi.items_vendor_discount), 0) AS discount_ratio,
#
#                                     avg(oi.order_hour)                                           AS avg_ordering_hour
#                              FROM KF.fact_order_items oi
#                                       INNER JOIN KF.plane_products pp ON oi.shop_product_id = pp.shop_product_id
#                              WHERE oi.is_gross = 1
#                                AND oi.order_date > today() - {TRAIN_DAYS}
#                                AND pp.category_level1_activity_type_id IN (1, 29)
#                                AND pp.category_level2_activity_type_id IN (1, 29)
#                                AND pp.product_id <> 0
#                              GROUP BY pp.brand_id, pp.category_level2_id),
#
#      product_seller_stats AS (SELECT pp.brand_id                        AS brand_id,
#                                      pp.category_level2_id              AS category_id,
#                                      countDistinct(pp.shop_product_id)  AS num_products,
#                                      countDistinct(pp.merchant_shop_id) AS num_sellers
#                               FROM KF.plane_products pp
#                               WHERE pp.category_level1_activity_type_id IN (1, 29)
#                                 AND pp.category_level2_activity_type_id IN (1, 29)
#                                 AND pp.product_id <> 0
#                               GROUP BY pp.brand_id, pp.category_level2_id),
#
#      first_appearance AS (SELECT pp.brand_id             AS brand_id,
#                                  pp.category_level2_id   AS category_id,
#                                  min(isph.snapshot_date) AS first_seen_date
#                           FROM KF.in_stock_price_history isph
#                                    INNER JOIN KF.plane_products pp ON isph.shop_product_id = pp.shop_product_id
#                           WHERE pp.category_level1_activity_type_id IN (1, 29)
#                             AND pp.category_level2_activity_type_id IN (1, 29)
#                             AND pp.product_id <> 0
#                           GROUP BY pp.brand_id, pp.category_level2_id)
#
# /*----- main SELECT: include all features -----*/
# SELECT concat(toString(b.brand_id), '|', toString(b.category_id))                                  AS brand_category_id,
#        toString(b.brand_id)                                                                        AS brand_id,
#        toString(b.category_id)                                                                     AS category_id,
#        toString(b.root_category_id)                                                                AS root_category_id,
#
#        -- availability & price distribution
#        ap.avg_availability,
#        ap.std_availability,
#        ap.avg_price,
#        ap.std_price,
#        ap.min_price,
#        ap.max_price,
#        ap.p25_price,
#        ap.p50_price,
#        ap.p75_price,
#
#        -- popularity & discount
#        p.total_popularity,
#        p.recent_popularity,
#        p.popularity_trend,
#        p.discount_ratio AS item_discount_ratio,
#        p.avg_ordering_hour AS item_avg_ordering_hour,
#
#        -- product/seller variety
#        ps.num_products,
#        ps.num_sellers,
#
#        -- item age (days since first appearance)
#        dateDiff('day', fa.first_seen_date, today())                                                AS item_age_days,
#
#        -- category‑level averages
#        avg(ap.avg_availability) OVER (PARTITION BY b.category_id)                                  AS category_availability,
#        avg(ap.avg_price) OVER (PARTITION BY b.category_id)                                         AS category_price,
#        avg(p.total_popularity) OVER (PARTITION BY b.category_id)                                   AS category_popularity,
#
#        -- brand‑level averages
#        avg(ap.avg_availability) OVER (PARTITION BY b.brand_id)                                     AS brand_availability,
#        avg(ap.avg_price) OVER (PARTITION BY b.brand_id)                                            AS brand_price,
#        avg(p.total_popularity) OVER (PARTITION BY b.brand_id)                                      AS brand_popularity,
#
#        -- brand‑category scores (compared to category average)
#        ap.avg_availability / nullIf(avg(ap.avg_availability) OVER (PARTITION BY b.category_id), 0) AS brand_category_availability_score,
#        ap.avg_price / nullIf(avg(ap.avg_price) OVER (PARTITION BY b.category_id), 0)               AS brand_category_price_score,
#        p.total_popularity / nullIf(avg(p.total_popularity) OVER (PARTITION BY b.category_id), 0)   AS brand_category_popularity_score
#
# FROM item_base b
#          JOIN availability_price ap
#               ON b.brand_id = ap.brand_id AND b.category_id = ap.category_id
#          JOIN popularity_discount p
#               ON b.brand_id = p.brand_id AND b.category_id = p.category_id
#          JOIN product_seller_stats ps
#               ON b.brand_id = ps.brand_id AND b.category_id = ps.category_id
#          JOIN first_appearance fa
#               ON b.brand_id = fa.brand_id AND b.category_id = fa.category_id
# """
#
# USER_SQL = f"""
# WITH
# -- per‑user basic stats ------------------------------------------------
# user_base AS (SELECT oi.user_id                                                                           AS user_id,
#                      nullIf(argMax(ua.polygon_id, ua.address_id), 0)                                      AS polygon_id,
#
#                      dateDiff('day', max(oi.order_date), today())                                         AS recency,
#                      dateDiff('day', min(oi.order_date), today())                                         AS length,
#                      countDistinct(oi.order_id)                                                           AS frequency,
#                      sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS monetary,
#
#                      countDistinct(pp.brand_id)                                                           AS brand_diversity,
#                      countDistinct(pp.category_level2_id)                                                 AS category_diversity,
#                      countDistinct(pp.category_level1_id)                                                 AS root_category_diversity,
#
#                      monetary / frequency                                                                 AS aov,
#                      count(oi.item_id) / frequency                                                        AS lpo, -- lines per order (number of item rows)
#                      sum(oi.quantity) / countDistinct(oi.order_id)                                        AS ipo, -- items per order (total quantity)
#
#                      avg(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS avg_item_spending,
#                      min(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS min_item_spending,
#                      max(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)            AS max_item_spending,
#                      quantile(0.25)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount) AS p25_item_spending,
#                      quantile(0.5)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount)  AS p50_item_spending,
#                      quantile(0.75)(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount) AS p75_item_spending,
#
#                      sum(oi.items_jet_discount + oi.brand_discount + oi.items_vendor_discount) /
#                      nullIf(sum(oi.payable_price + oi.items_voucher_discount + oi.items_jet_discount +
#                                 oi.brand_discount + oi.items_vendor_discount), 0)                         AS discount_ratio,
#
#                      avg(oi.order_hour)                                                                   AS avg_ordering_hour
#
#               FROM KF.fact_order_items AS oi
#                        JOIN KF.dim_merchant_shops AS ms ON oi.merchant_shop_id = ms.merchant_shop_id
#                        JOIN KF.dim_user_addresses AS ua ON oi.address_id = ua.address_id
#                        JOIN KF.plane_products AS pp ON oi.shop_product_id = pp.shop_product_id
#               WHERE oi.is_gross = 1
#                 AND oi.order_date > today() - {TRAIN_DAYS}
#                 AND ms.activity_type_id IN (1, 29)
#                 AND pp.category_level1_activity_type_id IN (1, 29)
#                 AND pp.category_level2_activity_type_id IN (1, 29)
#               GROUP BY oi.user_id)
#
# SELECT ub.user_id           AS user_id,
#        ub.polygon_id,
#        ub.recency,
#        ub.frequency,
#        ub.monetary,
#        ub.length,
#
#        ub.brand_diversity,
#        ub.category_diversity,
#        ub.root_category_diversity,
#
#        ub.avg_ordering_hour AS user_avg_ordering_hour,
#
#        ub.ipo,
#        ub.lpo,
#        ub.aov,
#
#        ub.avg_item_spending,
#        ub.min_item_spending,
#        ub.max_item_spending,
#        ub.p25_item_spending,
#        ub.p50_item_spending,
#        ub.p75_item_spending,
#
#        ub.discount_ratio    AS user_discount_ratio
# FROM user_base AS ub
# """

INTERACTION_SQL = f"""
SELECT oi.order_date                                                       AS order_date,
       oi.order_hour                                                       AS order_hour,
       oi.user_id                                                          AS user_id,
       oi.items_jet_discount                                               AS jet_discount,
       oi.items_voucher_discount                                           AS voucher_discount,
       concat(toString(pp.brand_id), '|', toString(pp.category_level2_id)) AS brand_category_id
FROM KF.fact_order_items oi
         INNER JOIN KF.plane_products pp
                    ON oi.shop_product_id = pp.shop_product_id
WHERE oi.is_gross = 1
  AND oi.order_date > today() - {TRAIN_DAYS}
  AND pp.category_level1_activity_type_id IN (1, 29)
  AND pp.category_level2_activity_type_id IN (1, 29)
  AND pp.product_id != 0
"""
#
# # ============================================================
# # 5) READ DATA
# # ============================================================
# item_df = client.query_df(ITEM_SQL)
# user_df = client.query_df(USER_SQL)
inter_events = client.query_df(INTERACTION_SQL)
#
# print("item_df shape      :", item_df.shape)
# print("user_df shape      :", user_df.shape)
# print("inter_events shape :", inter_events.shape)

In [178]:
# item_df.to_csv('librec_two_tower_category_brand_items_df_v2.csv')
# user_df.to_csv('librec_two_tower_category_brand_user_df_v2.csv')
# inter_events.to_csv('librec_two_tower_category_brand_inter_events_v2.csv')

In [185]:
item_df = pd.read_csv('./sample_data/librec_two_tower_category_brand_items_df_v2.csv', index_col=0).rename(columns={'brand_category_id': 'item'})
user_df = pd.read_csv('./sample_data/librec_two_tower_category_brand_user_df_v2.csv', index_col=0).rename(columns={'user_id': 'user'})
inter_events = pd.read_csv('./sample_data/librec_two_tower_category_brand_inter_events_v2.csv', index_col=0).rename(
	columns={'brand_category_id': 'item', 'user_id': 'user', 'order_date': 'time'})

In [186]:
inter_events['time'] = pd.to_datetime(inter_events['time']).astype('int64') // 10 ** 9
inter_events = inter_events.sort_values(by='time')

In [197]:
inter_events = inter_events[(inter_events['jet_discount'] == 0) & ((inter_events['voucher_discount'] == 0))]

In [157]:
print("item_df features:\n", item_df.columns.tolist(), "\n")
print("user_df features:\n", user_df.columns.tolist(), "\n")
print("inter_events features:\n", inter_events.columns.tolist(), "\n")

item_df features:
 ['item', 'brand_id', 'category_id', 'root_category_id', 'avg_availability', 'std_availability', 'avg_price', 'std_price', 'min_price', 'max_price', 'p25_price', 'p50_price', 'p75_price', 'total_popularity', 'recent_popularity', 'popularity_trend', 'item_discount_ratio', 'item_avg_ordering_hour', 'num_products', 'num_sellers', 'item_age_days', 'category_availability', 'category_price', 'category_popularity', 'brand_availability', 'brand_price', 'brand_popularity', 'brand_category_availability_score', 'brand_category_price_score', 'brand_category_popularity_score'] 

user_df features:
 ['user', 'polygon_id', 'recency', 'frequency', 'monetary', 'length', 'brand_diversity', 'category_diversity', 'root_category_diversity', 'user_avg_ordering_hour', 'ipo', 'lpo', 'aov', 'avg_item_spending', 'min_item_spending', 'max_item_spending', 'p25_item_spending', 'p50_item_spending', 'p75_item_spending', 'user_discount_ratio'] 

inter_events features:
 ['time', 'order_hour', 'user', 

In [212]:
inter_events = inter_events.groupby(['user', 'item'])['order_hour'].count().rename('label').reset_index()

In [213]:
filtered_users = inter_events.groupby('user')['item'].count().reset_index()
filtered_users = filtered_users[filtered_users['item'] > 10]

filtered_items = inter_events.groupby('item')['user'].count().reset_index()
filtered_items = filtered_items[filtered_items['user'] > 10]

In [214]:
inter_events = inter_events.loc[
    (inter_events['user'].isin(filtered_users['user'])) &
    (inter_events['item'].isin(filtered_items['item']))
]

In [215]:
inter_events['user'] = inter_events['user'].astype(str)
inter_events['item'] = inter_events['item'].astype(str)

item_df['item'] = item_df['item'].astype(str)
user_df['user'] = user_df['user'].astype(str) 

In [ ]:
df = (
	inter_events.
	merge(user_df, how="inner", on="user", ).
	merge(item_df, how="inner", on="item", )
)

df.loc[df['label'] > 10, 'label'] = 10

# 1 + 2 + ... + n = (n + 1) * n / 2
df['label'] = (df['label'] * (df['label'] + 1)) / 2

In [235]:
# import pandas as pd
#
# # Count interactions per user and item
# item_counts = df['item'].value_counts()
#
# # Apply filtering thresholds
# df = df[
# 	(df['item'].map(item_counts) >= 90)
# ].copy()

In [ ]:
# Single sparse columns
single_sparse_col = [
	"brand_id",
	"category_id",
	"root_category_id",
	# "polygon_id"
]

# Multi sparse columns (list of lists for multi‑hot encoding)
multi_sparse_col = [
	[]
]

# All dense features (both user and item sides, plus order_hour)
dense_col = [
	# Item dense features
	# 'avg_availability', 'std_availability', 'avg_price', 'std_price',
	# 'min_price', 'max_price', 'p25_price', 'p50_price', 'p75_price',
	# 'total_popularity', 'recent_popularity', 'popularity_trend',
	# 'item_discount_ratio', 'item_avg_ordering_hour', 'num_products', 'num_sellers',
	# 'item_age_days',
	# 'category_availability', 'category_price', 'category_popularity',
	# 'brand_availability', 'brand_price', 'brand_popularity',
	# 'brand_category_availability_score', 'brand_category_price_score',
	# 'brand_category_popularity_score',

	# User dense features
	# "recency", "length", "frequency", "monetary",
	# "brand_diversity", "category_diversity", "root_category_diversity",
	# "user_avg_ordering_hour", "ipo", "lpo", "aov",
	# "avg_item_spending", "min_item_spending", "max_item_spending",
	# "p25_item_spending", "p50_item_spending", "p75_item_spending",
	# "user_discount_ratio",

	# Interaction‑level feature
	# "order_hour",
	# "time"
]

# Features belonging to the user side
user_col = [
	# User sparse
	# "polygon_id",

	# User dense
	# "recency", "length", "frequency", "monetary",
	# "brand_diversity", "category_diversity", "root_category_diversity",
	# "user_avg_ordering_hour", "ipo", "lpo", "aov",
	# "avg_item_spending", "min_item_spending", "max_item_spending",
	# "p25_item_spending", "p50_item_spending", "p75_item_spending",
	# "user_discount_ratio",

	# Interaction‑level (treated as user feature for simplicity)
	# "order_hour",
	# "time"
	# User multi‑sparse
	# "category_id1", "category_id2", "category_id3", "category_id4", "category_id5",
	# "category_id6", "category_id7", "category_id8", "category_id9", "category_id10",
	# "brand_id1", "brand_id2", "brand_id3", "brand_id4", "brand_id5",
	# "brand_id6", "brand_id7", "brand_id8", "brand_id9", "brand_id10"
]

# Features belonging to the item side
item_col = [
	# Item sparse
	"brand_id",
	"category_id",
	"root_category_id",

	# Item dense
	# 'avg_availability', 'std_availability', 'avg_price', 'std_price',
	# 'min_price', 'max_price', 'p25_price', 'p50_price', 'p75_price',
	# 'total_popularity', 'recent_popularity', 'popularity_trend',
	# 'item_discount_ratio', 'item_avg_ordering_hour', 'num_products', 'num_sellers',
	# 'item_age_days',
	# 'category_availability', 'category_price', 'category_popularity',
	# 'brand_availability', 'brand_price', 'brand_popularity',
	# 'brand_category_availability_score', 'brand_category_price_score',
	# 'brand_category_popularity_score'
]

df = df[['user', 'item', 'label'] + user_col + item_col]
df = df.fillna(0)

In [237]:
 # Handle single sparse columns
for c in single_sparse_col:
	for df in [df]:
		df[c] = df[c].astype(str).fillna("missing")
		df.loc[df[c] == "0", c] = "missing"

# # Handle multi sparse columns
# multi_sparse_flat = [col for group in multi_sparse_col for col in group]
# for c in multi_sparse_flat:
# 	for df in [df]:
# 		df[c] = df[c].astype(str).fillna("missing")
# 		df.loc[df[c] == "0", c] = "missing"

# scaler = MinMaxScaler()
# df[dense_col] = scaler.fit_transform(df[dense_col]).astype(np.float32)

In [238]:
from recora.data import random_split

# train_data, eval_data = split_by_ratio_chrono(df, order=True, test_size=0.2)
train_data, eval_data = random_split(df, shuffle=True, test_size=0.05)

In [239]:
train_set, data_info = DatasetFeat.build_trainset(
	train_data=train_data,
	user_col=user_col,
	item_col=item_col,
	sparse_col=single_sparse_col,
	dense_col=dense_col,
	# multi_sparse_col=multi_sparse_col,
	# pad_val=["missing", "missing"],
	# TODO: important
	unique_feat=False
)

# train_set, data_info = DatasetFeat.build_trainset(
# 	train_data=train_data,
# 	user_col=[c for c in user_col],
# 	# item_col=item_col,
# 	sparse_col=[c for c in single_sparse_col if c not in item_col],
# 	dense_col=[c for c in dense_col if c not in item_col],
# 	shuffle=False,
# 	# multi_sparse_col=[c for c in multi_sparse_col if c not in item_col],
# 	# pad_val=["missing", "missing"],
# 	# TODO: important
# 	unique_feat=False
# )

eval_set = DatasetFeat.build_testset(eval_data)
# #
# # print(data_info)

In [247]:
from recora.data import DatasetPure

train_set, data_info = DatasetPure.build_trainset(train_data)
eval_set = DatasetPure.build_evalset(eval_data)

# del train_data, eval_data

In [266]:
def reset_state(name):
	tf.compat.v1.reset_default_graph()
	print("\n", "=" * 30, name, "=" * 30)


metrics = ["rmse", "mae", "r2"]

# ranking negative sampling
NEG_SAMPLING = False
NEG_SAMPLER = "unconsumed"  # random | unconsumed | popular
NUM_NEG = 5
BATCH_SIZE = 2048

# Convert to dict (libreco expects a dict)
config_dict = {
	'gpu_options': {
		'allow_growth': True,
		'per_process_gpu_memory_fraction': 0.95  # Reduce to 60% to leave room for the OS
	}
}

from recora.algorithms import (
	YouTubeRetrieval,
	YouTubeRanking,
	ItemCF,
	UserCF,
	FM,
	AutoInt,
	WideDeep,
	ItemCF
)

# reset_state("FM")
# model = FM(
# 	"rating",
# 	data_info,
# 	loss_type="focal",
# 	embed_size=32,
# 	n_epochs=5,
# 	lr=1e-3,
# 	lr_decay=False,
# 	reg=None,
# 	batch_size=BATCH_SIZE,
# 	num_neg=1,
# 	use_bn=True,
# 	dropout_rate=0.2,
# 	tf_sess_config=config_dict,
# 	lower_upper_bound=(1,55)
# )

# reset_state("DeepFM")
# model = DeepFM(
# 	"rating",
# 	data_info,
# 	loss_type="focal",
# 	embed_size=4,
# 	n_epochs=3,
# 	lr=1e-4,
# 	lr_decay=False,
# 	reg=None,
# 	batch_size=BATCH_SIZE,
# 	num_neg=1,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(64, 32),
# 	tf_sess_config=config_dict,
# 	lower_upper_bound=(1,55)
# )

# reset_state("WideDeep")
# model = WideDeep(
# 	"rating",
# 	data_info,
# 	loss_type="cross_entropy",
# 	embed_size=8,
# 	n_epochs=5,
# 	lr={"wide": 0.01, "deep": 1e-4},
# 	lr_decay=False,
# 	reg=None,
# 	batch_size=BATCH_SIZE,
# 	num_neg=1,
# 	use_bn=False,
# 	dropout_rate=None,
# 	hidden_units=(128, 64, 32),
# 	tf_sess_config=config_dict,
# 	lower_upper_bound=(1,5)
# )
# reset_state("AutoInt")
# model = AutoInt(
# 	"rating",
# 	data_info,
# 	embed_size=32,
# 	n_epochs=2,
# 	att_embed_size=(8, 8, 8),
# 	num_heads=4,
# 	use_residual=False,
# 	lr=1e-3,
# 	lr_decay=False,
# 	reg=None,
# 	batch_size=BATCH_SIZE,
# 	num_neg=1,
# 	use_bn=False,
# 	dropout_rate=None,
# 	tf_sess_config=config_dict,
# 	lower_upper_bound=(1,55)
# )

# reset_state("YoutubeRetrieval")
# model = YouTubeRetrieval(
# 	"ranking",
# 	data_info,
# 	loss_type="sampled_softmax",
# 	embed_size=32,
# 	norm_embed=False,
# 	n_epochs=10,
# 	lr=1.25*1e-3,
# 	lr_decay=True,
# 	reg=1e-6,
# 	batch_size=BATCH_SIZE,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	num_sampled_per_batch=BATCH_SIZE * NUM_NEG,
# 	sampler='log_uniform_candidate_sampler',
# 	recent_num=31,
# 	random_num=31
# 	# multi_sparse_combiner='normal'
# )

# reset_state("YouTubeRanking")
# model = YouTubeRanking(
# 	"ranking",
# 	data_info,
# 	loss_type="cross_entropy",
# 	embed_size=8,
# 	n_epochs=10,
# 	lr=2e-3,
# 	lr_decay=False,
# 	reg=1e-5,
# 	batch_size=BATCH_SIZE,
# 	num_neg=NUM_NEG,
# 	sampler=NEG_SAMPLER,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(512, 256, 128),
# 	tf_sess_config=config_dict,
# 	recent_num=5,
# )

reset_state("ItemCF")
model = ItemCF(
	task="rating",
	data_info=data_info,
	k_sim=20,
	sim_type="pearson",
	mode="invert",
	num_threads=1,
	min_common=1,
	lower_upper_bound=(1, 55)
)

model.fit(
	train_set,
	neg_sampling=NEG_SAMPLING,

	verbose=2,
	# shuffle=True,

	metrics=metrics,

	eval_data=eval_set,
	eval_user_num=5000,
	k=TOP_K,
	eval_batch_size=BATCH_SIZE,
	# num_workers=2
)


 ============================== ItemCF ==============================
Training start time: 2026-04-06 02:37:27
Final block size and num: (3608, 1)
sim_matrix elapsed: 0.830s
sim_matrix, shape: (3608, 3608), num_elements: 3519362, density: 27.0353 %


eval_pointwise:   2%|▏         | 1/50 [00:00<00:05,  9.69it/s]

No common interaction or similar neighbor for user 25948 and item 1439, proceed with default prediction
No common interaction or similar neighbor for user 25948 and item 2307, proceed with default prediction
No common interaction or similar neighbor for user 27981 and item 1882, proceed with default prediction
No common interaction or similar neighbor for user 27981 and item 90, proceed with default prediction
No common interaction or similar neighbor for user 17890 and item 2574, proceed with default prediction
No common interaction or similar neighbor for user 17890 and item 992, proceed with default prediction


eval_pointwise: 100%|██████████| 50/50 [00:04<00:00, 11.60it/s]

	 eval rmse: 2.8307
	 eval mae: 1.8812
	 eval r2: -31.0523


In [267]:
from recora.evaluation import evaluate, print_metrics

eval_result = evaluate(
    model=model,
    data=eval_set,
    neg_sampling=True,
    eval_batch_size=BATCH_SIZE,
    k=5,
    metrics=metrics,
    sample_user_num=5000,
)

eval_pointwise: 100%|██████████| 50/50 [00:04<00:00, 12.00it/s]


In [268]:
eval_result

{'rmse': 2.8307362017200286,
 'mae': 1.8811534703729051,
 'r2': -31.052269774913334}

In [269]:
# model = YouTubeRetrieval(
# 	"ranking",
# 	data_info,
# 	loss_type="sampled_softmax",
# 	embed_size=32,
# 	norm_embed=False,
# 	n_epochs=10,
# 	lr=1.25*1e-3,
# 	lr_decay=True,
# 	reg=1e-6,
# 	batch_size=512,
# 	use_bn=False,
# 	dropout_rate=0.2,
# 	hidden_units=(256, 128, 64),
# 	tf_sess_config=config_dict,
# 	num_sampled_per_batch=512 * 15,
# 	recent_num=31,
# 	random_num=31
# 	# multi_sparse_combiner='normal'
# )

# Epoch 10 elapsed: 178.519s
# 	 train_loss: 5.8241
# eval_pointwise: 100%|██████████| 3788/3788 [00:01<00:00, 3569.97it/s]
# eval_listwise: 100%|██████████| 5000/5000 [00:05<00:00, 950.57it/s]
# 	 eval log_loss: 0.2397
# 	 eval balanced_accuracy: 0.9053
# 	 eval roc_auc: 0.9718
# 	 eval pr_auc: 0.9684
# 	 eval precision@20: 0.0295
# 	 eval recall@20: 0.2478
# 	 eval map@20: 0.1391
# 	 eval ndcg@20: 0.2134

In [ ]:
from libreco.data import DataInfo

SAVE_PATH = 'youtube-retr'
SAVE_MODEL_NAME = 'youtuberetrieval'
# save data_info, specify model save folder
data_info.save(
	path=SAVE_PATH,
	model_name=SAVE_MODEL_NAME
)
# set manual=True will use `numpy` to save model
# set manual=False will use `tf.train.Saver` to save model
# set inference=True will only save the necessary variables for prediction and recommendation
model.save(
	path=SAVE_PATH,
	model_name=SAVE_MODEL_NAME,
	manual=False,
	inference_only=True
)

In [270]:
result = model.recommend_user(
	"30953879",
	n_rec=1000,
	filter_consumed=False,
	# user_feats={
	# 	"order_hour": 15 / 23
	# }
)

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result["30953879"]
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result[result['category_id'] == 651].head(20)

,30953879,brand_id,category_id,category_title,brand_title
2,16923012651,16923,651,سیگار,کریستال سلکت
24,11053012651,11053,651,سیگار,اونیکس
37,16817012651,16817,651,سیگار,کارلو
69,1447012651,1447,651,سیگار,بهمن
74,12627012651,12627,651,سیگار,کاوالو
93,17589012651,17589,651,سیگار,مونتانا
95,520012651,520,651,سیگار,کریستال
105,17568012651,17568,651,سیگار,مور
112,17677012651,17677,651,سیگار,ناسو
132,19008012651,19008,651,سیگار,ترا


In [256]:
result = train_data.loc[train_data['user'] == "30953879"].sort_values(by='label', ascending=False).reset_index()
result[['brand_id', 'category_id']] = (
	result['item']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('./sample_data/brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result.loc[result['category_id'] == 651, ['user', 'item', 'label', 'category_title', 'brand_title']]

,user,item,label,category_title,brand_title
0,30953879,12627012651,55.0,سیگار,کاوالو
1,30953879,7311012651,45.0,سیگار,وینستون
2,30953879,16843012651,21.0,سیگار,کالوالو
4,30953879,11653012651,3.0,سیگار,بیستون
9,30953879,14051012651,1.0,سیگار,اسی
16,30953879,1447012651,1.0,سیگار,بهمن
18,30953879,12257012651,1.0,سیگار,کمل


In [ ]:
import tensorflow as tf
from libreco.data import DataInfo
from libreco.algorithms import TwoTower, YouTubeRetrieval

# important to reset graph if model is loaded in the same shell.
tf.compat.v1.reset_default_graph()
# load data_info
data_info = DataInfo.load(
	SAVE_PATH, model_name=SAVE_MODEL_NAME
)
print(data_info)
# load model, should specify the model name, e.g., DeepFM
model: YouTubeRetrieval = YouTubeRetrieval.load(
	path=SAVE_PATH, model_name=SAVE_MODEL_NAME, data_info=data_info, manual=True
)

In [ ]:
# model.user_embeds_np[data_info.user2id[30953879]].shape
model.user_embeds_np.shape

In [ ]:
import faiss
import numpy as np
import pandas as pd

faiss_index: faiss.swigfaiss.IndexIVFFlat = faiss.read_index('./embed_model/faiss_index.bin')
user_embedding = model.get_user_embedding(30953879, include_bias=True).astype(np.float32).reshape(1, -1)
distances, ids = faiss_index.search(user_embedding, 100)

In [ ]:
distances

In [ ]:
result = pd.DataFrame(list(map(lambda item: list(map(lambda x: int(x), item.split("012"))), [data_info.id2item[x] for x in ids.ravel().tolist()])),
                      columns=['brand_id', 'category_id'])
result['distances'] = distances.ravel()

brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')

In [ ]:
# result[result['category_id'] == 69]
result

# Init `embed_deploy`

In [ ]:
from libserving.serialization import embed2redis, save_embed, save_faiss_index

path = "embed_model"  # specify model saving directory
save_embed(path, model)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis, user_consumed2redis, user_embed2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_embed.json")) as f:
		user_embeds = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, embed) in enumerate(user_embeds.items()):
		pipe.hset("user_embed", u, ujson.dumps(embed))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_embed2redis ...")

In [ ]:
from libserving.serialization import save_faiss_index

print(path, model)
save_faiss_index(path, model)

In [ ]:
import requests
import json

payload = {
	"user": "30953879",
	"n_rec": 100
}

result = requests.post(
	"http://127.0.0.1:8000/embed/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

In [ ]:
result[['brand_id', 'category_id']] = (
	result['item']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')

# Init `online_deploy`

In [ ]:
from libserving.serialization import save_online, online2redis

path = "online_model"  # specify model saving directory
save_online(path, model, version=1)  # save model in json format

In [ ]:
from libserving.serialization.redis import redis_connection, model_name2redis, id_mapping2redis
import ujson
import os

host = "localhost"
port = 6379
db = 0
chunk_size = 10000  # ← same as you used before

# ------------------------------------------------------------------
# 1. model_name + id_mapping (small, unchanged)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	model_name2redis(path, r)
	print("\nmodel_name2redis ...")

with redis_connection(host, port, db) as r:
	id_mapping2redis(path, r)
	print("\nid_mapping2redis ...")

# ------------------------------------------------------------------
# 2. user_consumed (your existing chunked version)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	with open(os.path.join(path, "user_consumed.json")) as f:
		data = ujson.load(f)

	pipe = r.pipeline()
	for i, (u, items) in enumerate(data.items()):
		pipe.hset("user_consumed", u, ujson.dumps(items))
		if (i + 1) % chunk_size == 0:
			pipe.execute()
			pipe = r.pipeline()
	pipe.execute()
	print("\nuser_consumed2redis ...")

# ------------------------------------------------------------------
# 3. features2redis → FULLY CHUNKED (the heavy part for online models)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	feature_path = os.path.join(path, "features.json")
	with open(feature_path) as f:
		feats = ujson.load(f)

	r.set("n_users", feats["n_users"])
	r.set("n_items", feats["n_items"])
	if "max_seq_len" in feats:
		r.set("max_seq_len", feats["max_seq_len"])

	# user_sparse_values (per-user)
	if "user_sparse_col_index" in feats:
		r.hset("feature", "user_sparse", 1)
		r.set("user_sparse_col_index", ujson.dumps(feats["user_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_sparse_values"]):
			pipe.hset("user_sparse_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_sparse_values (list push)
	if "item_sparse_col_index" in feats:
		r.hset("feature", "item_sparse", 1)
		r.set("item_sparse_col_index", ujson.dumps(feats["item_sparse_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_sparse_values"]):
			pipe.rpush("item_sparse_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# user_dense_values (per-user)
	if "user_dense_col_index" in feats:
		r.hset("feature", "user_dense", 1)
		r.set("user_dense_col_index", ujson.dumps(feats["user_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["user_dense_values"]):
			pipe.hset("user_dense_values", str(i), ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	# item_dense_values (list push)
	if "item_dense_col_index" in feats:
		r.hset("feature", "item_dense", 1)
		r.set("item_dense_col_index", ujson.dumps(feats["item_dense_col_index"]))
		pipe = r.pipeline()
		for i, vals in enumerate(feats["item_dense_values"]):
			pipe.rpush("item_dense_values", ujson.dumps(vals))
			if (i + 1) % chunk_size == 0:
				pipe.execute()
				pipe = r.pipeline()
		pipe.execute()

	print("\nfeatures2redis ...")

# ------------------------------------------------------------------
# 4. user_sparse2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_sparse_fields_path = os.path.join(path, "user_sparse_fields.json")
	if os.path.exists(user_sparse_fields_path):
		with open(user_sparse_fields_path) as f:
			user_sparse_fields = ujson.load(f)
		r.hset("user_sparse_fields", mapping=user_sparse_fields)

	user_sparse_idx_mapping_path = os.path.join(path, "user_sparse_idx_mapping.json")
	if os.path.exists(user_sparse_idx_mapping_path):
		with open(user_sparse_idx_mapping_path) as f:
			user_sparse_idx_mapping = ujson.load(f)
		for col, idx_mapping in user_sparse_idx_mapping.items():
			col_name = f"user_sparse_idx_mapping__{col}"
			r.hset(col_name, mapping=idx_mapping)

	print("\nuser_sparse2redis ...")

# ------------------------------------------------------------------
# 5. user_dense2redis (small, no chunk needed)
# ------------------------------------------------------------------
with redis_connection(host, port, db) as r:
	user_dense_fields_path = os.path.join(path, "user_dense_fields.json")
	if os.path.exists(user_dense_fields_path):
		with open(user_dense_fields_path) as f:
			user_dense_fields = ujson.load(f)
		r.hset("user_dense_fields", mapping=user_dense_fields)

	print("\nuser_dense2redis ...")

print("\n✅ All online2redis data loaded into Redis with chunking!")

In [ ]:
from libserving.serialization import save_faiss_index

save_faiss_index(path, model)

In [ ]:
result = model.recommend_user(
	30953879,
	n_rec=10,
	filter_consumed=False,
	# user_feats={
	# 	"order_hour": 20
	# }
)

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result[30953879]
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)
# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

In [ ]:
json.dumps(payload)

In [ ]:
import requests
import json

payload = {
	"user": 30953879,
	"n_rec": 10,
	"filter_consumed": False,
	"return_scores": False,
	"user_feats": {
		"order_hour": 12
	}
}

result = requests.post(
	"http://127.0.0.1:8000/online/recommend",
	headers={"Content-Type": "application/json"},
	data=json.dumps(payload)
).json()

result = pd.DataFrame(result)
result[['brand_id', 'category_id']] = (
	result['recommendations']
	.str.split("012", expand=True, n=1)  # n=1 ensures we only split once
)

# Optional: convert to integer if they are numeric IDs
result['brand_id'] = pd.to_numeric(result['brand_id'], errors='coerce')
result['category_id'] = pd.to_numeric(result['category_id'], errors='coerce')
brand_category_titles = pd.read_csv('brand_category_titles.csv')
result = result.merge(brand_category_titles, on=['brand_id', 'category_id'], how='left')
result

### Read `brand_category_titles` data

In [ ]:
BRAND_CATEGORY_SQL = """
SELECT category_level2_id                                        AS category_id,
       brand_id                                                  AS brand_id,
       dictGet('KF.dim_categories', 'title', category_level2_id) AS category_title,
       dictGet('KF.dim_brands', 'title', brand_id)               AS brand_title
FROM KF.plane_products
GROUP BY category_level2_id, brand_id
"""

brand_category_titles = client.query_df(BRAND_CATEGORY_SQL)

In [ ]:
brand_category_titles.to_csv('brand_category_titles.csv', index=False)

In [ ]:
brand_category_titles = pd.read_csv('brand_category_titles.csv')